In [ ]:
# 查看原始 MQTT 消息格式
import json
import paho.mqtt.client as mqtt
import time

received_messages = []

def on_message(client, userdata, msg):
    payload = json.loads(msg.payload.decode("utf-8"))
    received_messages.append(payload)
    print(f"原始数据: {payload}")

client = mqtt.Client()
client.on_message = on_message
client.connect("10.22.9.10", 1883, 60)
client.subscribe("quest/data")
client.loop_start()

time.sleep(10)
client.loop_stop()
client.disconnect()

print(f"\n接收到 {len(received_messages)} 条消息")
if received_messages:
    print(f"数据格式: {received_messages[0]}")


In [ ]:
# 创建 Quest3sController，测量获取数据的频率
import sys
sys.path.insert(0, '../')
from controllers.quest3s.quest3s import Quest3sController

controller = Quest3sController()
controller.connect()
print("✓ Connected to Quest3s\n")

import time
start_time = time.time()
count = 0

print("测量 get_action() 频率 (采集5秒数据)...\n")
try:
    while time.time() - start_time < 30:
        action = controller.get_action()
        if action is not None:
            count += 1
            elapsed = time.time() - start_time
            fps = count / elapsed
            print(f"[{elapsed:.2f}s] Frame {count}: freq={fps:.1f}Hz, pos={action['position']}, btn={action['buttons']}")
except KeyboardInterrupt:
    pass
finally:
    elapsed = time.time() - start_time
    fps = count / elapsed if elapsed > 0 else 0
    print(f"\n统计结果:")
    print(f"  采集时间: {elapsed:.2f}s")
    print(f"  收到帧数: {count}")
    print(f"  平均频率: {fps:.1f} Hz")
    controller.disconnect()
    print("✓ Disconnected")


In [3]:
import json
import socket

HOST = "172.30.109.22"
PORT = 16001
GROUP = 1


def read_line(sock, buf=b""):
    while b"\n" not in buf:
        chunk = sock.recv(65536)
        if not chunk:
            raise ConnectionError("socket closed")
        buf += chunk
    idx = buf.index(b"\n")
    line = buf[:idx].rstrip(b"\r")
    return json.loads(line.decode()), buf[idx + 1:]


with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(5)
    s.connect((HOST, PORT))
    s.sendall(b'{"Communication": "FRC_Connect"}\r\n')
    data = json.loads(s.recv(4096).decode())
    dynamic_port = data["PortNumber"]
    print("FRC_Connect:", data)

buf = b""
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.settimeout(5)
    s.connect((HOST, dynamic_port))

    for pkt in [
        {"Command": "FRC_Initialize", "GroupMask": GROUP},
        {"Command": "FRC_SetUFrameUTool", "UFrameNumber": 0, "UToolNumber": 1, "Group": GROUP},
        {"Command": "FRC_ReadCartesianPosition", "Group": GROUP},
    ]:
        s.sendall((json.dumps(pkt) + "\r\n").encode())
        resp, buf = read_line(s, buf)
        print("RESP:", resp)
        if pkt["Command"] == "FRC_ReadCartesianPosition":
            first = resp

    p = first["Position"]
    cfg = first["Configuration"]

    motion = {
        "Instruction": "FRC_LinearMotion",
        "SequenceID": 1,
        "Configuration": cfg,
        "Position": {
            "X": p["X"] + 10.0,
            "Y": p["Y"],
            "Z": p["Z"],
            "W": p["W"],
            "P": p["P"],
            "R": p["R"],
            "Ext1": 0.0,
            "Ext2": 0.0,
            "Ext3": 0.0,
        },
        "SpeedType": "mmSec",
        "Speed": 10,
        "TermType": "CNT",
        "TermValue": 100,
    }

    print("SEND MOTION:", motion)
    s.sendall((json.dumps(motion) + "\r\n").encode())

    resp, buf = read_line(s, buf)
    print("MOTION RESP:", resp)


FRC_Connect: {'Communication': 'FRC_Connect', 'ErrorID': 0, 'PortNumber': 16002, 'MajorVersion': 5, 'MinorVersion': 0}
RESP: {'Command': 'FRC_Initialize', 'ErrorID': 0, 'GroupMask': 1}
RESP: {'Command': 'FRC_SetUFrameUTool', 'ErrorID': 0, 'Group': 1}
RESP: {'Command': 'FRC_ReadCartesianPosition', 'ErrorID': 0, 'TimeTag': 4027275, 'Group': 1, 'Configuration': {'UToolNumber': 1, 'UFrameNumber': 0, 'Front': 1, 'Up': 1, 'Left': 0, 'Flip': 0, 'Turn4': 0, 'Turn5': 0, 'Turn6': 0}, 'Position': {'X': 422.167, 'Y': 12.779, 'Z': -55.148, 'W': 178.773, 'P': 3.456, 'R': -179.264, 'Ext1': 0.0, 'Ext2': 0.0, 'Ext3': 0.0}}
SEND MOTION: {'Instruction': 'FRC_LinearMotion', 'SequenceID': 1, 'Configuration': {'UToolNumber': 1, 'UFrameNumber': 0, 'Front': 1, 'Up': 1, 'Left': 0, 'Flip': 0, 'Turn4': 0, 'Turn5': 0, 'Turn6': 0}, 'Position': {'X': 432.167, 'Y': 12.779, 'Z': -55.148, 'W': 178.773, 'P': 3.456, 'R': -179.264, 'Ext1': 0.0, 'Ext2': 0.0, 'Ext3': 0.0}, 'SpeedType': 'mmSec', 'Speed': 10, 'TermType': 'CN

TimeoutError: timed out